> **Superseded (2026-11-25).** Kept for the record of how the project developed. The current pipeline is `ev_siting_final.ipynb` (EV-registration demand weight, audit fixes, all models and scenarios in one notebook). Numbers here use the earlier population-based weight.

# EV Charging Optimizer (Delhi): Model 2, Weighted K-means

Demand-driven model. Where the grid baseline places stations by land use, this one places them where **demand is concentrated**.

**Pipeline**
1. Load the Delhi boundary and the demand layer `demand_weights` (one row per demand point, with a weight)
2. Clean the weights (drop empty / invalid rows, optional rescale to a control total)
3. Choose the number of stations `k` (fixed to match the grid baseline, or by silhouette score)
4. Fit **weighted K-means** on projected coordinates (metres), so each centroid is the demand-weighted centre of its cluster
5. Optional: pull centroids that landed in low-demand spots toward the densest point of their cluster
6. Snap centroids to the nearest feasible site (one-to-one), flag any that cannot be snapped
7. Evaluate with the common metrics (coverage within R, distance statistics) so results line up with the grid and MCLP models
8. Sensitivity: k sweep, random-seed stability, demand-weight perturbation (site persistence)
9. Export in the same format as the grid baseline

**Status:** template. `CFG["use_synthetic"] = True` runs the whole notebook on made-up data. Set it to `False` and fill in the paths once the real demand layer exists.

The one variable the rest of the notebook depends on is **`demand_weights`**: a GeoDataFrame of Point geometries in EPSG:32643 with a numeric column named by `CFG["weight_col"]`.

## 0. Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, Point, Polygon
from scipy.spatial import cKDTree
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Config and data contract

| Input | Format | Needs |
|---|---|---|
| Ward boundaries | geojson | same file as the grid notebook (geometry only) |
| **`demand_weights`** | point file (gpkg / geojson) | one row per demand point, Point geometry, numeric weight column (`weight_col`) |
| Feasible sites (optional) | point file | parking lots, fuel stations, commercial plots, etc. |
| Existing chargers (optional) | point file | only used to report how the current network scores on the same metrics |

All geometry work happens in **EPSG:32643 (UTM 43N)**, so K-means distances are in metres. Do not run K-means on raw lat/lon.

**What the weights mean.** `weight` is the demand at that point (for example the share of Delhi's four-wheeler EV demand allocated to a 1 km cell). Only the relative pattern matters for placement, so the absolute level is not important, but the weights must be non-negative and comparable across points. Building them is a separate step; this notebook consumes the result.

**Choosing `k`.** `k_mode="fixed"` uses `k_fixed`, which should equal the grid baseline's station count so that the three models are compared at the same budget. `k_mode="silhouette"` picks the best `k` from the sweep instead.

In [ ]:
CFG = dict(
    # --- data ---
    wards_path="data/DELHI Ward Boundaries and Land Use.geojson",
    demand_path="data/delhi_demand_weights.gpkg",        # the demand_weights layer
    feasible_sites_path=None,                            # e.g. "data/feasible_sites.geojson"; None = raw centroids
    chargers_path=None,                                  # e.g. "data/delhi_ev_chargers_master.geojson"; None = skip
    use_synthetic=True,                                  # flip to False once real data is in place

    # --- geometry ---
    crs_metric="EPSG:32643",
    boundary_close_m=300,
    weight_col="weight",                                 # column holding the demand weight
    control_total=None,                                  # e.g. projected 4W EV count; None = keep weights as given

    # --- choosing k ---
    k_mode="fixed",                                      # "fixed" or "silhouette"
    k_fixed=102,                                         # PLACEHOLDER: set to the grid baseline's station count
    k_sweep=list(range(10, 151, 10)),                    # candidates for the sweep / silhouette
    silhouette_sample=5000,                              # silhouette is O(n^2); score a random subsample

    # --- K-means ---
    n_init=20,
    max_iter=500,
    seed=42,

    # --- optional low-demand relocation (variant, off by default) ---
    relocate_low_demand=False,
    local_radius_m=500,                                  # radius used to measure demand around a centroid
    low_demand_frac=0.05,                                # relocate if < 5% of the cluster's demand lies within the radius

    # --- placement ---
    snap_tolerance_m=500,                                # max distance to move a centroid to a feasible site

    # --- evaluation ---
    R_list_m=[1000, 2000, 2100, 3000],                   # coverage radii (2100 m ~ worst case of the 3 km grid)
    R_main_m=2100,

    # --- sensitivity ---
    n_seeds=10,
    mc_runs=20,
    mc_sigma=0.30,                                       # lognormal noise on weights
    persistence_delta_m=1000,                            # a base site "persists" if a rerun has a site within this distance
)

## 2. Data loading

The boundary loader is the same as in the grid notebook. The synthetic generator makes a fake Delhi-sized city with a few demand hotspots, so every cell below runs before the real data exists.

In [ ]:
def union(gs):
    """Union of a GeoSeries (works on old and new GeoPandas)."""
    return gs.union_all() if hasattr(gs, "union_all") else gs.unary_union


def load_boundary(path, crs, close_m=300):
    """Dissolve wards into one polygon, close slivers, fill interior holes."""
    w = gpd.read_file(path).to_crs(crs)
    b = union(w.geometry).buffer(close_m).buffer(-close_m)
    parts = list(getattr(b, "geoms", [b]))
    b = union(gpd.GeoSeries([Polygon(p.exterior) for p in parts], crs=crs))
    return gpd.GeoDataFrame(geometry=[b], crs=crs)


def load_points(path, crs):
    return gpd.read_file(path).to_crs(crs)[["geometry"]].reset_index(drop=True)


def load_demand(path, crs, weight_col):
    d = gpd.read_file(path).to_crs(crs)
    if weight_col not in d.columns:
        raise KeyError(f"demand layer has no column '{weight_col}'. Columns: {list(d.columns)}")
    d = d.rename(columns={weight_col: "weight"})[["weight", "geometry"]]
    d["geometry"] = d.geometry.centroid                  # accept polygon cells as well as points
    return d.reset_index(drop=True)

In [ ]:
def make_synthetic(crs, seed=0, cell_m=1000):
    """Fake city: 40 x 44 km box, demand on a 1 km lattice built from a few Gaussian hotspots."""
    rng = np.random.default_rng(seed)
    cx, cy = 715_000, 3_165_000
    boundary = gpd.GeoDataFrame(geometry=[box(cx - 20_000, cy - 22_000, cx + 20_000, cy + 22_000)], crs=crs)

    xs = np.arange(cx - 20_000 + cell_m / 2, cx + 20_000, cell_m)
    ys = np.arange(cy - 22_000 + cell_m / 2, cy + 22_000, cell_m)
    X, Y = np.meshgrid(xs, ys)
    X, Y = X.ravel(), Y.ravel()

    hubs = [(-3000, 2000, 4500, 1.0), (4000, -3000, 5000, 0.8), (-9000, -8000, 4000, 0.6),
            (8000, 9000, 3500, 0.7), (0, 12000, 3000, 0.4)]
    dens = np.zeros_like(X)
    for hx, hy, s, a in hubs:
        dens += a * np.exp(-((X - (cx + hx)) ** 2 + (Y - (cy + hy)) ** 2) / (2 * s ** 2))
    dens = dens * rng.lognormal(0, 0.35, len(X)) + 0.02          # background + noise
    w = 1000 * dens / dens.sum()                                  # pretend control total = 1,000 vehicles

    demand_weights = gpd.GeoDataFrame({"weight": w}, geometry=gpd.points_from_xy(X, Y), crs=crs)
    sites = gpd.GeoDataFrame(
        geometry=[Point(rng.uniform(cx - 20_000, cx + 20_000), rng.uniform(cy - 22_000, cy + 22_000))
                  for _ in range(600)], crs=crs)
    existing = gpd.GeoDataFrame(
        geometry=[Point(rng.uniform(cx - 15_000, cx + 15_000), rng.uniform(cy - 15_000, cy + 15_000))
                  for _ in range(150)], crs=crs)
    return boundary, demand_weights, sites, existing


if CFG["use_synthetic"]:
    boundary, demand_weights, feasible, existing = make_synthetic(CFG["crs_metric"])
else:
    boundary = load_boundary(CFG["wards_path"], CFG["crs_metric"], CFG["boundary_close_m"])
    demand_weights = load_demand(CFG["demand_path"], CFG["crs_metric"], CFG["weight_col"])
    feasible = load_points(CFG["feasible_sites_path"], CFG["crs_metric"]) if CFG["feasible_sites_path"] else None
    existing = load_points(CFG["chargers_path"], CFG["crs_metric"]) if CFG["chargers_path"] else None

if "weight" not in demand_weights.columns:                    # synthetic path already uses "weight"
    demand_weights = demand_weights.rename(columns={CFG["weight_col"]: "weight"})

print(f"Boundary area: {boundary.area.iloc[0] / 1e6:,.0f} km2 | demand points: {len(demand_weights)} | "
      f"feasible sites: {0 if feasible is None else len(feasible)} | "
      f"existing chargers: {0 if existing is None else len(existing)}")

## 3. Clean the weights

K-means with sample weights needs strictly positive, finite weights. Rows with zero, negative or missing weight carry no demand, so they are dropped (and counted). If `control_total` is set, weights are rescaled so they sum to it; this does not change where stations go, only the units of `cluster_demand`.

In [ ]:
def clean_weights(d, control_total=None):
    n0 = len(d)
    w = pd.to_numeric(d["weight"], errors="coerce")
    ok = np.isfinite(w) & (w > 0) & d.geometry.notna() & ~d.geometry.is_empty
    out = d[ok].copy()
    out["weight"] = w[ok].astype(float)
    if control_total is not None:
        out["weight"] *= control_total / out["weight"].sum()
    print(f"Weights: kept {len(out)} of {n0} points ({n0 - len(out)} dropped) | "
          f"total {out['weight'].sum():,.1f} | median {out['weight'].median():.3g} | max {out['weight'].max():.3g}")
    return out.reset_index(drop=True)


def xy(d):
    return np.column_stack([d.geometry.x.to_numpy(), d.geometry.y.to_numpy()])


demand = clean_weights(demand_weights, CFG["control_total"])
X, w = xy(demand), demand["weight"].to_numpy()

## 4. Weighted K-means

Objective minimised: **Σ wᵢ · ‖xᵢ − c(i)‖²** (weighted squared distance to the assigned centre). Each centre ends up at the demand-weighted mean of its cluster, so dense areas pull centres toward them and receive more stations.

Two things to remember when reading results:
- squared distance punishes far-away demand heavily, so K-means favours a low *average* distance rather than a guaranteed coverage radius;
- a centroid is an average, so it can land where nobody lives or nothing can be built. Section 6 (snapping) deals with the second point.

In [ ]:
def fit_weighted_kmeans(X, w, k, seed=42, n_init=20, max_iter=500):
    km = KMeans(n_clusters=k, init="k-means++", n_init=n_init, max_iter=max_iter, random_state=seed)
    km.fit(X, sample_weight=w)
    return km


def nearest_dist(points_xy, sites_xy):
    d, _ = cKDTree(sites_xy).query(points_xy)
    return d


def wquantile(values, weights, q):
    o = np.argsort(values)
    v, ww = values[o], weights[o]
    c = np.cumsum(ww) / ww.sum()
    return float(v[np.searchsorted(c, q)])


def evaluate_layout(demand_xy, demand_w, site_xy, R_list, name="layout"):
    """Common yardstick, reused by all three models: coverage within R and distance statistics (demand-weighted)."""
    d = nearest_dist(demand_xy, site_xy)
    row = {
        "layout": name,
        "stations": len(site_xy),
        "mean_dist_m": float(np.average(d, weights=demand_w)),
        "median_dist_m": wquantile(d, demand_w, 0.5),
        "p95_dist_m": wquantile(d, demand_w, 0.95),
        "max_dist_m": float(d.max()),
    }
    for R in R_list:
        row[f"cov_{R}m_pct"] = round(100 * demand_w[d <= R].sum() / demand_w.sum(), 1)
    return {k: (round(v, 1) if isinstance(v, float) else v) for k, v in row.items()}

## 5. Choosing k

The sweep fits K-means for each candidate `k` and records inertia (elbow), silhouette score and the weighted mean distance. **Caveat:** `silhouette_score` has no sample-weight option, so it treats every demand point equally; use it as a rough guide, not as the final word. For the three-model comparison, `k_mode="fixed"` (same budget for every model) is the cleaner choice.

In [ ]:
def sweep_k(X, w, ks, cfg):
    rows = []
    rng = np.random.default_rng(cfg["seed"])
    for k in ks:
        if k >= len(X):
            continue
        km = fit_weighted_kmeans(X, w, k, cfg["seed"], max(3, cfg["n_init"] // 4), cfg["max_iter"])
        sil = silhouette_score(X, km.labels_, sample_size=min(len(X), cfg["silhouette_sample"]),
                               random_state=cfg["seed"])
        d = nearest_dist(X, km.cluster_centers_)
        rows.append({"k": k, "inertia": km.inertia_, "silhouette": round(float(sil), 3),
                     "mean_dist_m": round(float(np.average(d, weights=w)), 0),
                     f"cov_{cfg['R_main_m']}m_pct": round(100 * w[d <= cfg["R_main_m"]].sum() / w.sum(), 1)})
    return pd.DataFrame(rows)


def choose_k(sweep, cfg):
    if cfg["k_mode"] == "fixed":
        return int(cfg["k_fixed"])
    if cfg["k_mode"] == "silhouette":
        return int(sweep.loc[sweep["silhouette"].idxmax(), "k"])
    raise ValueError("k_mode must be 'fixed' or 'silhouette'")


sweep = sweep_k(X, w, CFG["k_sweep"], CFG)
k = choose_k(sweep, CFG)
print(f"k_mode = {CFG['k_mode']} -> k = {k}")
sweep

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].plot(sweep["k"], sweep["inertia"], marker="o"); axes[0].set_title("Elbow (weighted inertia)")
axes[1].plot(sweep["k"], sweep["silhouette"], marker="o", color="tab:green"); axes[1].set_title("Silhouette (unweighted)")
axes[2].plot(sweep["k"], sweep[f"cov_{CFG['R_main_m']}m_pct"], marker="o", color="tab:red")
axes[2].set_title(f"Weighted coverage within {CFG['R_main_m']} m (%)")
for ax in axes:
    ax.set_xlabel("k"); ax.grid(alpha=0.3); ax.axvline(k, color="grey", ls="--", lw=0.8)
plt.tight_layout(); plt.show()

## 6. From centroids to stations

1. **Raw centroids** from the fitted model, with each cluster's demand and size.
2. **Optional relocation** (`relocate_low_demand`): a centroid with very little demand around it is moved to the highest-demand-weighted medoid among the cluster's heaviest points. This is a simple stand-in for the "avoid centroids near low-weighted points" idea in the custom K-means literature, not a re-implementation of that paper's penalty.
3. **Snapping**: each centroid is matched **one-to-one** to a feasible site within `snap_tolerance_m` (Hungarian assignment on distances, so two centroids can never claim the same site). Centroids with no feasible site in range keep their raw position and are flagged `snapped = False`.

In [ ]:
def build_stations(km, X, w, crs):
    labels = km.labels_
    c = km.cluster_centers_
    cluster_demand = np.bincount(labels, weights=w, minlength=len(c))
    n_points = np.bincount(labels, minlength=len(c))
    st = gpd.GeoDataFrame(
        {"station_id": np.arange(len(c)),
         "cluster_demand": cluster_demand,
         "n_points": n_points,
         "demand_share": cluster_demand / w.sum(),
         "raw_x": c[:, 0], "raw_y": c[:, 1]},
        geometry=gpd.points_from_xy(c[:, 0], c[:, 1]), crs=crs)
    st["relocated"] = False
    return st


def relocate_low_demand(st, X, w, labels, radius_m, frac, top_n=200):
    """Move centroids with little demand nearby to the weighted medoid of their cluster's heaviest points."""
    st = st.copy()
    xs, ys = st.geometry.x.to_numpy().copy(), st.geometry.y.to_numpy().copy()
    for i in st.index[st["n_points"] > 0]:
        m = labels == st.at[i, "station_id"]
        Xi, wi = X[m], w[m]
        near = np.hypot(Xi[:, 0] - xs[i], Xi[:, 1] - ys[i]) <= radius_m
        if wi[near].sum() >= frac * wi.sum():
            continue
        cand = np.argsort(wi)[-top_n:]
        cost = [(wi * np.hypot(Xi[:, 0] - Xi[j, 0], Xi[:, 1] - Xi[j, 1])).sum() for j in cand]
        best = cand[int(np.argmin(cost))]
        xs[i], ys[i] = Xi[best]
        st.at[i, "relocated"] = True
    st = st.set_geometry(gpd.points_from_xy(xs, ys, crs=st.crs))
    return st


def snap_to_sites(st, feasible, tol_m):
    """One-to-one snap of stations to feasible sites within tol_m. Unmatched stations keep their position."""
    st = st.copy()
    st["snapped"] = False
    st["snap_dist_m"] = 0.0
    if feasible is None or len(feasible) == 0 or len(st) == 0:
        return st
    S = xy(st)
    F = xy(feasible)
    dist = np.hypot(S[:, None, 0] - F[None, :, 0], S[:, None, 1] - F[None, :, 1])
    BIG = 1e9
    cost = np.where(dist <= tol_m, dist, BIG)
    r, c = linear_sum_assignment(cost)
    ok = cost[r, c] < BIG
    r, c = r[ok], c[ok]
    new_x, new_y = S[:, 0].copy(), S[:, 1].copy()
    new_x[r], new_y[r] = F[c, 0], F[c, 1]
    st = st.set_geometry(gpd.points_from_xy(new_x, new_y, crs=st.crs))
    st.loc[st.index[r], "snapped"] = True
    st.loc[st.index[r], "snap_dist_m"] = dist[r, c]
    return st

## 7. Model runner

In [ ]:
def run_kmeans_model(demand, X, w, k, feasible, cfg):
    km = fit_weighted_kmeans(X, w, k, cfg["seed"], cfg["n_init"], cfg["max_iter"])
    raw = build_stations(km, X, w, demand.crs)
    st = raw
    if cfg["relocate_low_demand"]:
        st = relocate_low_demand(st, X, w, km.labels_, cfg["local_radius_m"], cfg["low_demand_frac"])
    final = snap_to_sites(st, feasible, cfg["snap_tolerance_m"])
    return dict(k=k, km=km, raw=raw, stations=final)


def summarise_run(res, X, w, cfg, existing=None):
    rows = [evaluate_layout(X, w, xy(res["raw"]), cfg["R_list_m"], "kmeans: raw centroids"),
            evaluate_layout(X, w, xy(res["stations"]), cfg["R_list_m"], "kmeans: after snapping")]
    if existing is not None and len(existing) > 0:
        rows.append(evaluate_layout(X, w, xy(existing), cfg["R_list_m"], "existing network"))
    out = pd.DataFrame(rows).set_index("layout")
    out["pct_snapped"] = [np.nan, round(100 * res["stations"]["snapped"].mean(), 1)] + ([np.nan] if len(rows) > 2 else [])
    return out

## 8. Run the model

In [ ]:
res = run_kmeans_model(demand, X, w, k, feasible, CFG)
summary = summarise_run(res, X, w, CFG, existing)
summary.T

In [ ]:
res["stations"].drop(columns="geometry").describe().round(2).T

In [ ]:
def plot_result(res, demand, boundary, title=None):
    st = res["stations"]
    fig, ax = plt.subplots(figsize=(8, 9))
    boundary.boundary.plot(ax=ax, color="black", linewidth=1)
    sc = ax.scatter(demand.geometry.x, demand.geometry.y, c=demand["weight"], s=6, cmap="Greys", alpha=0.7)
    plt.colorbar(sc, ax=ax, shrink=0.6, label="demand weight")
    ax.scatter(res["raw"].geometry.x, res["raw"].geometry.y, s=16, facecolors="none",
               edgecolors="tab:blue", label="raw centroid", zorder=4)
    ok = st["snapped"]
    ax.scatter(st[~ok].geometry.x, st[~ok].geometry.y, s=16, color="crimson", label="not snapped", zorder=5)
    ax.scatter(st[ok].geometry.x, st[ok].geometry.y, s=16, color="tab:orange", label="snapped to feasible site", zorder=6)
    ax.set_title(title or f"Weighted K-means | k = {res['k']}")
    ax.legend(loc="lower left"); ax.set_aspect("equal"); ax.set_axis_off()
    plt.show()

plot_result(res, demand, boundary)

## 9. Sensitivity

**9a. Random-seed stability.** K-means finds a local optimum, so different seeds give different layouts. We compare each rerun with the seed-0 layout: the distance from each baseline station to the nearest rerun station (small = the site is stable), plus the objective value.

In [ ]:
def seed_stability(X, w, k, cfg, n_seeds):
    base = fit_weighted_kmeans(X, w, k, cfg["seed"], cfg["n_init"], cfg["max_iter"])
    rows = []
    for s in range(1, n_seeds + 1):
        r = fit_weighted_kmeans(X, w, k, cfg["seed"] + s, cfg["n_init"], cfg["max_iter"])
        d = nearest_dist(base.cluster_centers_, r.cluster_centers_)
        rows.append({"seed": cfg["seed"] + s, "inertia": r.inertia_, "inertia_vs_base_pct": round(100 * (r.inertia_ / base.inertia_ - 1), 2),
                     "median_shift_m": round(float(np.median(d)), 0), "p90_shift_m": round(float(np.quantile(d, 0.9)), 0)})
    return pd.DataFrame(rows)

stab = seed_stability(X, w, k, CFG, CFG["n_seeds"])
stab

**9b. Demand-weight perturbation.** The weights are an estimate, so we multiply each by lognormal noise (`mc_sigma`), refit, and ask: *how often does each baseline site reappear?* This is the persistence idea from the literature review (the share of runs in which a site is chosen). A site counts as "persisting" in a run if the rerun has a station within `persistence_delta_m` of it. Each rerun layout is also scored on the **base** weights, to show how much coverage is lost if the true demand differs from the estimate.

In [ ]:
def weight_perturbation(demand, X, w, base_res, cfg):
    rng = np.random.default_rng(cfg["seed"] + 1000)
    base_xy = xy(base_res["stations"])
    hit = np.zeros(len(base_xy))
    rows = []
    for i in range(cfg["mc_runs"]):
        wp = w * rng.lognormal(0.0, cfg["mc_sigma"], len(w))
        km = fit_weighted_kmeans(X, wp, base_res["k"], cfg["seed"] + i, max(3, cfg["n_init"] // 4), cfg["max_iter"])
        d_site = nearest_dist(base_xy, km.cluster_centers_)
        hit += (d_site <= cfg["persistence_delta_m"])
        d = nearest_dist(X, km.cluster_centers_)
        rows.append({"run": i, f"cov_{cfg['R_main_m']}m_on_base_w_pct": round(100 * w[d <= cfg["R_main_m"]].sum() / w.sum(), 1),
                     "mean_dist_on_base_w_m": round(float(np.average(d, weights=w)), 0)})
    persistence = hit / cfg["mc_runs"]
    return pd.DataFrame(rows), persistence

mc, persistence = weight_perturbation(demand, X, w, res, CFG)
res["stations"]["persistence"] = persistence
print(f"Median site persistence: {np.median(persistence):.2f} | sites persisting in >=80% of runs: "
      f"{(persistence >= 0.8).mean():.0%}")
mc.describe().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
boundary.boundary.plot(ax=ax, color="black", linewidth=1)
st = res["stations"]
sc = ax.scatter(st.geometry.x, st.geometry.y, c=st["persistence"], cmap="RdYlGn", vmin=0, vmax=1, s=40, edgecolors="k", linewidths=0.4)
plt.colorbar(sc, ax=ax, shrink=0.6, label=f"share of runs with a site within {CFG['persistence_delta_m']} m")
ax.set_title("Site persistence under demand-weight noise"); ax.set_aspect("equal"); ax.set_axis_off()
plt.show()

## 10. Export

Same layout as the grid baseline: a CSV with `lon` / `lat`, a GeoJSON, plus the summary and sensitivity tables.

In [ ]:
os.makedirs("outputs", exist_ok=True)

out = res["stations"].copy()
out["model"] = "kmeans"
out = out.to_crs("EPSG:4326")
out["lon"], out["lat"] = out.geometry.x, out.geometry.y
tag = f"kmeans_k{res['k']}"
out.drop(columns="geometry").to_csv(f"outputs/{tag}_stations.csv", index=False)
out.to_file(f"outputs/{tag}_stations.geojson", driver="GeoJSON")
summary.to_csv(f"outputs/{tag}_summary.csv")
sweep.to_csv(f"outputs/{tag}_k_sweep.csv", index=False)
stab.to_csv(f"outputs/{tag}_seed_stability.csv", index=False)
mc.to_csv(f"outputs/{tag}_weight_perturbation.csv", index=False)
print("Saved to outputs/")

## Open items before real runs

- [ ] Build `demand_weights` (point layer, EPSG:32643, weight column) and set `demand_path` / `weight_col`
- [ ] Set `k_fixed` to the grid baseline's station count (3 km, 1% threshold gives 102 candidate cells; 13 if only unserved cells are counted) so both models share the same budget
- [ ] Decide whether the final comparison uses a feasible-sites layer for snapping, and which tolerance is realistic
- [ ] Check demand-point resolution: K-means is O(n·k) per iteration, so 1 km points for Delhi (about 1,500) are trivial, building-level points would not be
- [ ] Euclidean distance is used throughout; consider road-network distance for the final evaluation
- [ ] Existing chargers are only scored, not used as fixed centres; a variant that keeps them and adds new ones is a later iteration
- [ ] Use the same `evaluate_layout` function in the MCLP notebook so the three models are scored identically